# LSTM Model Screening

Trains and evaluates the longitudinal LSTM baseline and saves its test predictions.

This is the publication copy for the SAGE-Credit repository. The original analysis logic is preserved, machine-specific paths are replaced with portable configuration, and cached outputs are cleared.

> Download the Home Credit Default Risk competition files separately; the dataset is not redistributed here.

In [ ]:
from pathlib import Path
import os

# Portable repository configuration. Override either location with an environment variable.
WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
DATA_DIR = Path(os.environ.get("HOME_CREDIT_DATA_DIR", PROJECT_ROOT / "data" / "raw")).expanduser().resolve()
ARTIFACT_DIR = Path(os.environ.get("SAGE_CREDIT_ARTIFACT_DIR", PROJECT_ROOT / "artifacts")).expanduser().resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {DATA_DIR}")
print(f"Artifact directory: {ARTIFACT_DIR}")


In [ ]:
import os
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping

LOOKBACK_MONTHS = 60
master = pd.read_pickle(ARTIFACT_DIR / "home_credit_master_60m_REAL_NOW.pkl")

def clean():
  gc.collect()


In [ ]:
print(master.shape)

In [ ]:
count=0

for i in master.columns:
  count+=1

print(count)

In [ ]:
features = master.drop(columns=['SK_ID_CURR','TARGET','MONTHS_BALANCE']).columns

y = master.groupby('SK_ID_CURR', sort=False)['TARGET'].first().to_numpy(dtype=np.int32)

X = master[features].to_numpy(dtype=np.float32)
X = X.reshape(-1, LOOKBACK_MONTHS, len(features))

print(X.shape)
print(y.shape)

print(X.dtype, y.dtype)
print(X.nbytes / 1024**3, "GB")

In [ ]:
monthly_nonzero = master.groupby("MONTHS_BALANCE").apply(
  lambda x: (x.drop(columns=["SK_ID_CURR", "TARGET", "MONTHS_BALANCE"]) != 0).mean().mean()
)

print(monthly_nonzero)

In [ ]:
del master
clean()

In [ ]:
scaler = StandardScaler()

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.1, random_state=42,stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1111, random_state=42,stratify=y_train_val)

del X
del y
del X_train_val
del y_train_val
clean()

In [ ]:
print(X_train.shape)

In [ ]:
n_features = X_train.shape[2]

X_train = X_train.reshape(-1, n_features)
X_val = X_val.reshape(-1,n_features)
X_test = X_test.reshape(-1, n_features)

scaler.fit(X_train)

X_train = scaler.transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

X_train = X_train.reshape(-1, LOOKBACK_MONTHS, n_features)
X_val = X_val.reshape(-1, LOOKBACK_MONTHS, n_features)
X_test = X_test.reshape(-1, LOOKBACK_MONTHS, n_features)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

clean()

In [ ]:
model = Sequential()

model.add(Input(shape=(LOOKBACK_MONTHS,n_features)))
model.add(LSTM(128))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['auc'])
early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)

history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=2048,callbacks=[early_stop])

In [ ]:
pred_probs = model.predict(X_val).flatten()

threshold = 0
best_threshold = 0
best_f1 = 0

for threshold in np.arange(0,1,0.01):
  pred_labels = (pred_probs >= threshold).astype(int)
  f1 = f1_score(y_val, pred_labels)
  if f1 > best_f1:
    best_f1 = f1
    best_threshold = threshold

print(best_threshold, best_f1)

In [ ]:
pred_labels = (pred_probs >= best_threshold).astype(int)

print(accuracy_score(y_val, pred_labels))
print(f1_score(y_val, pred_labels))
print(recall_score(y_val, pred_labels))
print(precision_score(y_val, pred_labels))
print(roc_auc_score(y_val, pred_probs))

In [ ]:
test_probs = model.predict(X_test).flatten()
test_labels = (test_probs >= best_threshold).astype(int)

print("TEST RESULTS")
print(accuracy_score(y_test, test_labels))
print(f1_score(y_test, test_labels))
print(recall_score(y_test, test_labels))
print(precision_score(y_test, test_labels))
print(roc_auc_score(y_test, test_probs))

In [ ]:
from sklearn.metrics import roc_curve, auc, RocCurveDisplay

RocCurveDisplay.from_predictions(y_test, model.predict(X_test))
plt.show()

In [ ]:
rnn_probs = model.predict(X_test).ravel()

np.save(ARTIFACT_DIR / "rnn_probs.npy", rnn_probs)
np.save(ARTIFACT_DIR / "rnn_y_test.npy", y_test)

#**ALL PLOTS**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

master = pd.read_pickle(ARTIFACT_DIR / "home_credit_master_60m_REAL_NOW.pkl")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13, 5)
)

# Training and validation loss
axes[0].plot(
    history.history["loss"],
    linewidth=2,
    label="Training Loss"
)

axes[0].plot(
    history.history["val_loss"],
    linewidth=2,
    label="Validation Loss"
)

axes[0].set_title("Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(alpha=0.25)

# Training and validation ROC-AUC
axes[1].plot(
    history.history["auc"],
    linewidth=2,
    label="Training ROC-AUC"
)

axes[1].plot(
    history.history["val_auc"],
    linewidth=2,
    label="Validation ROC-AUC"
)

axes[1].set_title("Training and Validation ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("ROC-AUC")
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.suptitle(
    "LSTM Training History",
    fontsize=16,
    y=1.02
)

plt.tight_layout()
plt.show()